In [15]:
# ============================================================
# F7 — Week 05 Black-Box Optimisation
# Structure: Config → Load → Surrogate → Acquisition → Diagnostics → Output
# Goal: MAXIMISE y
# X: 6D in [0,1], y: scalar
# ============================================================

# ---------------------------
# Imports
# ---------------------------
import os
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

from scipy.stats import norm


In [16]:

# ---------------------------
# 0) Config
# ---------------------------
np.random.seed(42)

INPUTS_PATH  = "data/F7/initial_inputs.npy"
OUTPUTS_PATH = "data/F7/initial_outputs.npy"

# If you're running locally and the folders don't exist yet:
os.makedirs("data/F7", exist_ok=True)
os.makedirs("outputs/F7", exist_ok=True)

@dataclass
class BOConfig:
    # Candidate generation
    n_global: int = 8000
    n_local: int = 2500
    local_scales: tuple = (0.05, 0.05, 0.05, 0.05, 0.05, 0.05)  # tighten for refinement
    # Acquisition
    xi: float = 0.01              # small positive for exploitation bias
    # Diversity / anti-duplicate
    min_dist: float = 0.02        # reject candidates too close to existing points
    # Models
    rf_n_estimators: int = 600
    rf_min_samples_leaf: int = 2
    gbm_ensemble_size: int = 12
    gbm_max_depth: int = 3
    gbm_learning_rate: float = 0.05
    gbm_n_estimators: int = 450
    # Diagnostics
    cv_folds: int = 5

cfg = BOConfig()

In [18]:
# ---------------------------
# 1) Load
# ---------------------------
def safe_load_npy(path: str):
    if os.path.exists(path):
        return np.load(path)
    # fallback for environments where files are mounted elsewhere (optional)
    alt = "/mnt/data/" + os.path.basename(path)
    if os.path.exists(alt):
        return np.load(alt)
    raise FileNotFoundError(f"Could not find {path} (or {alt}).")

X = safe_load_npy(INPUTS_PATH)
y = safe_load_npy(OUTPUTS_PATH)

X = np.asarray(X, dtype=float)
y = np.asarray(y, dtype=float).reshape(-1)

assert X.ndim == 2 and X.shape[1] == 6, f"Expected X shape (n,6), got {X.shape}"
assert y.ndim == 1 and y.shape[0] == X.shape[0], f"Expected y shape (n,), got {y.shape}"

n, d = X.shape
y_best = float(np.max(y))
best_idx = int(np.argmax(y))
x_best = X[best_idx].copy()

print(f"Loaded dataset: n={n}, d={d}")
print(f"Best observed y: {y_best:.6f} at index {best_idx}")
print(f"Best observed x: {x_best}")


Loaded dataset: n=34, d=6
Best observed y: 1.565071 at index 30
Best observed x: [0.       0.425878 0.225301 0.207625 0.373905 0.758381]


In [19]:
# ---------------------------
# 2) Surrogate
# ---------------------------
# --- Gaussian Process (with standardisation) ---
x_scaler = StandardScaler()
y_scaler = StandardScaler()

Xs = x_scaler.fit_transform(X)
ys = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3)) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,
    normalize_y=False,
    n_restarts_optimizer=8,
    random_state=42
)
gp.fit(Xs, ys)

# --- Random Forest ---
rf = RandomForestRegressor(
    n_estimators=cfg.rf_n_estimators,
    min_samples_leaf=cfg.rf_min_samples_leaf,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y)

# --- GBM ensemble ---
gbm_ens = []
for i in range(cfg.gbm_ensemble_size):
    m = GradientBoostingRegressor(
        random_state=1000 + i,
        max_depth=cfg.gbm_max_depth,
        learning_rate=cfg.gbm_learning_rate,
        n_estimators=cfg.gbm_n_estimators
    )
    m.fit(X, y)
    gbm_ens.append(m)


# Predict + uncertainty helpers
def gp_predict_mu_sigma(Xcand):
    Xcand_s = x_scaler.transform(Xcand)
    mu_s, std_s = gp.predict(Xcand_s, return_std=True)
    mu = y_scaler.inverse_transform(mu_s.reshape(-1, 1)).ravel()
    # std transforms by multiplying scale factor
    std = std_s * float(y_scaler.scale_[0])
    std = np.maximum(std, 1e-12)
    return mu, std

def rf_predict_mu_sigma(Xcand):
    # tree-level predictions → mean & std
    all_tree_preds = np.stack([t.predict(Xcand) for t in rf.estimators_], axis=0)  # (n_trees, n_cand)
    mu = all_tree_preds.mean(axis=0)
    std = all_tree_preds.std(axis=0, ddof=1)
    std = np.maximum(std, 1e-12)
    return mu, std

def gbm_predict_mu_sigma(Xcand):
    preds = np.stack([m.predict(Xcand) for m in gbm_ens], axis=0)  # (n_models, n_cand)
    mu = preds.mean(axis=0)
    std = preds.std(axis=0, ddof=1)
    std = np.maximum(std, 1e-12)
    return mu, std


In [20]:
# ---------------------------
# 3) Acquisition (Expected Improvement, MAXIMISE)
# ---------------------------
def expected_improvement(mu, sigma, y_best, xi=0.0):
    # EI for maximisation
    # improvement = mu - y_best - xi
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    # if sigma is tiny, EI should be ~max(imp,0)
    ei = np.where(sigma <= 1e-12, np.maximum(imp, 0.0), ei)
    return np.maximum(ei, 0.0)

def min_distance_to_dataset(Xcand, Xdata):
    # Euclidean distance to nearest existing sample
    # returns vector of length len(Xcand)
    diffs = Xcand[:, None, :] - Xdata[None, :, :]
    dist = np.sqrt(np.sum(diffs**2, axis=2))
    return dist.min(axis=1)

def propose_candidate(X, y, cfg: BOConfig):
    y_best = float(np.max(y))
    x_best = X[int(np.argmax(y))].copy()

    # Global candidates
    X_global = np.random.rand(cfg.n_global, X.shape[1])

    # Local candidates around current best
    scales = np.array(cfg.local_scales).reshape(1, -1)
    X_local = x_best.reshape(1, -1) + np.random.normal(0.0, 1.0, size=(cfg.n_local, X.shape[1])) * scales
    X_local = np.clip(X_local, 0.0, 1.0)

    Xcand = np.vstack([X_global, X_local])

    # Diversity filter: remove points too close to existing data
    dmin = min_distance_to_dataset(Xcand, X)
    keep = dmin >= cfg.min_dist
    Xcand_f = Xcand[keep]
    dmin_f = dmin[keep]

    if Xcand_f.shape[0] < 200:
        # if too strict, relax slightly (without asking you)
        relaxed = max(cfg.min_dist * 0.5, 1e-6)
        keep = dmin >= relaxed
        Xcand_f = Xcand[keep]
        dmin_f = dmin[keep]

    # Model-wise EI
    mu_gp, sig_gp = gp_predict_mu_sigma(Xcand_f)
    mu_rf, sig_rf = rf_predict_mu_sigma(Xcand_f)
    mu_gb, sig_gb = gbm_predict_mu_sigma(Xcand_f)

    ei_gp = expected_improvement(mu_gp, sig_gp, y_best, xi=cfg.xi)
    ei_rf = expected_improvement(mu_rf, sig_rf, y_best, xi=cfg.xi)
    ei_gb = expected_improvement(mu_gb, sig_gb, y_best, xi=cfg.xi)

    # Blended EI (equal weights)
    ei_blend = (ei_gp + ei_rf + ei_gb) / 3.0

    best_cand_idx = int(np.argmax(ei_blend))
    x_next = Xcand_f[best_cand_idx].copy()

    details = {
        "x_next": x_next,
        "ei_blend_max": float(ei_blend[best_cand_idx]),
        "ei_gp": float(ei_gp[best_cand_idx]),
        "ei_rf": float(ei_rf[best_cand_idx]),
        "ei_gb": float(ei_gb[best_cand_idx]),
        "dmin_to_data": float(dmin_f[best_cand_idx]),
        "mu_gp": float(mu_gp[best_cand_idx]),
        "mu_rf": float(mu_rf[best_cand_idx]),
        "mu_gb": float(mu_gb[best_cand_idx]),
        "sig_gp": float(sig_gp[best_cand_idx]),
        "sig_rf": float(sig_rf[best_cand_idx]),
        "sig_gb": float(sig_gb[best_cand_idx]),
        "y_best": float(y_best),
    }
    return x_next, details

x_next, acq_details = propose_candidate(X, y, cfg)



In [21]:
# ---------------------------
# 4) Diagnostics
# ---------------------------
def crossval_report(model_name, predict_fn):
    # KFold CV: train each model type inside folds (quick, approximate diagnostics)
    kf = KFold(n_splits=min(cfg.cv_folds, len(y)), shuffle=True, random_state=42)
    preds = np.zeros_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr = y[train_idx]

        if model_name == "GP":
            xs = StandardScaler().fit(Xtr)
            ys_scaler = StandardScaler().fit(ytr.reshape(-1, 1))
            Xtr_s = xs.transform(Xtr)
            ytr_s = ys_scaler.transform(ytr.reshape(-1, 1)).ravel()

            kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3)) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
            m = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, random_state=42)
            m.fit(Xtr_s, ytr_s)

            mu_s = m.predict(xs.transform(Xte))
            mu = ys_scaler.inverse_transform(mu_s.reshape(-1, 1)).ravel()

        elif model_name == "RF":
            m = RandomForestRegressor(
                n_estimators=max(250, cfg.rf_n_estimators // 2),
                min_samples_leaf=cfg.rf_min_samples_leaf,
                random_state=42,
                n_jobs=-1
            )
            m.fit(Xtr, ytr)
            mu = m.predict(Xte)

        elif model_name == "GBM":
            # small ensemble inside fold (faster)
            ens = []
            for i in range(6):
                gm = GradientBoostingRegressor(
                    random_state=1000 + i,
                    max_depth=cfg.gbm_max_depth,
                    learning_rate=cfg.gbm_learning_rate,
                    n_estimators=max(250, cfg.gbm_n_estimators // 2)
                )
                gm.fit(Xtr, ytr)
                ens.append(gm)
            mu = np.mean([m.predict(Xte) for m in ens], axis=0)

        else:
            raise ValueError("Unknown model_name")

        preds[test_idx] = mu

    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)
    return rmse, r2, preds

rmse_gp, r2_gp, pred_gp_cv = crossval_report("GP", None)
rmse_rf, r2_rf, pred_rf_cv = crossval_report("RF", None)
rmse_gb, r2_gb, pred_gb_cv = crossval_report("GBM", None)

print("\nCV Diagnostics (approx):")
print(f"  GP  : RMSE={rmse_gp:.4f}, R2={r2_gp:.4f}")
print(f"  RF  : RMSE={rmse_rf:.4f}, R2={r2_rf:.4f}")
print(f"  GBM : RMSE={rmse_gb:.4f}, R2={r2_gb:.4f}")

# Predicted vs observed (in-sample for quick view)
mu_gp_all, _ = gp_predict_mu_sigma(X)
mu_rf_all, _ = rf_predict_mu_sigma(X)
mu_gb_all, _ = gbm_predict_mu_sigma(X)

def save_scatter(y_true, y_pred, title, path):
    plt.figure(figsize=(5.5, 5))
    plt.scatter(y_true, y_pred)
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx])
    plt.xlabel("Observed y")
    plt.ylabel("Predicted y")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

def save_residuals(y_true, y_pred, title, path):
    res = y_true - y_pred
    plt.figure(figsize=(6, 4))
    plt.scatter(y_pred, res)
    plt.axhline(0.0)
    plt.xlabel("Predicted y")
    plt.ylabel("Residual (obs - pred)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

save_scatter(y, mu_gp_all, "F7 GP: Predicted vs Observed (in-sample)", "outputs/F7/pred_vs_obs_gp.png")
save_scatter(y, mu_rf_all, "F7 RF: Predicted vs Observed (in-sample)", "outputs/F7/pred_vs_obs_rf.png")
save_scatter(y, mu_gb_all, "F7 GBM-Ens: Predicted vs Observed (in-sample)", "outputs/F7/pred_vs_obs_gbm.png")

save_residuals(y, mu_gp_all, "F7 GP: Residuals vs Pred", "outputs/F7/residuals_gp.png")
save_residuals(y, mu_rf_all, "F7 RF: Residuals vs Pred", "outputs/F7/residuals_rf.png")
save_residuals(y, mu_gb_all, "F7 GBM-Ens: Residuals vs Pred", "outputs/F7/residuals_gbm.png")

# Feature importances (RF)
rf_importances = rf.feature_importances_
plt.figure(figsize=(6, 3.8))
plt.bar(np.arange(d), rf_importances)
plt.xticks(np.arange(d), [f"x{i+1}" for i in range(d)])
plt.ylabel("Importance")
plt.title("F7 RF Feature Importances")
plt.tight_layout()
plt.savefig("outputs/F7/rf_feature_importances.png", dpi=160)
plt.close()

# Region / diversity checks
dmin_next = min_distance_to_dataset(x_next.reshape(1, -1), X)[0]
print("\nDiversity / region check:")
print(f"  Nearest distance from proposed x_next to dataset: {dmin_next:.6f}")
print(f"  Proposed x_next: {x_next}")

# Simple coverage check: per-dimension min/max + where x_next lies
mins = X.min(axis=0); maxs = X.max(axis=0)
print("\nPer-dimension coverage (min, max) and proposed x_next position:")
for i in range(d):
    print(f"  x{i+1}: [{mins[i]:.3f}, {maxs[i]:.3f}]  ->  x_next={x_next[i]:.3f}")




CV Diagnostics (approx):
  GP  : RMSE=0.2364, R2=0.6514
  RF  : RMSE=0.3614, R2=0.1854
  GBM : RMSE=0.3714, R2=0.1397

Diversity / region check:
  Nearest distance from proposed x_next to dataset: 0.035475
  Proposed x_next: [0.00325173 0.42921114 0.24132403 0.21339687 0.35439815 0.73458595]

Per-dimension coverage (min, max) and proposed x_next position:
  x1: [0.000, 0.942]  ->  x_next=0.003
  x2: [0.012, 0.982]  ->  x_next=0.429
  x3: [0.004, 0.925]  ->  x_next=0.241
  x4: [0.074, 0.961]  ->  x_next=0.213
  x5: [0.015, 0.999]  ->  x_next=0.354
  x6: [0.051, 0.951]  ->  x_next=0.735


In [22]:
# ---------------------------
# 5) Output
# ---------------------------
# Print one line (what you submit to tutor)
np.set_printoptions(suppress=True, precision=8)

print("\n====================================")
print("F7 Week 05 — Proposed next X (submit this one line)")
print("====================================")
print(x_next)

print("\nAcquisition details (blended EI):")
for k, v in acq_details.items():
    if k == "x_next":
        continue
    print(f"  {k}: {v}")

# Save proposed point
np.save("data/F7/proposed_next_x.npy", x_next.reshape(1, -1))
np.savetxt("data/F7/proposed_next_x.csv", x_next.reshape(1, -1), delimiter=",")
print("\nSaved:")
print("  data/F7/proposed_next_x.npy")
print("  data/F7/proposed_next_x.csv")
print("  outputs/F7/*.png (diagnostics plots)")


F7 Week 05 — Proposed next X (submit this one line)
[0.00325173 0.42921114 0.24132403 0.21339687 0.35439815 0.73458595]

Acquisition details (blended EI):
  ei_blend_max: 0.027762030147671674
  ei_gp: 0.01664672155535976
  ei_rf: 0.023364139184458524
  ei_gb: 0.04327522970319673
  dmin_to_data: 0.03547474290291677
  mu_gp: 1.5799671652950678
  mu_rf: 1.1324124627080772
  mu_gb: 1.6183460417757092
  sig_gp: 0.035250969334377706
  sig_rf: 0.3822823684733493
  sig_gb: 0.008696683673692775
  y_best: 1.5650708126009862

Saved:
  data/F7/proposed_next_x.npy
  data/F7/proposed_next_x.csv
  outputs/F7/*.png (diagnostics plots)
